# 04 — Task Lifecycle: States, Polling, Cancel, and Multi-Turn

## Why this notebook exists

In **notebook 03** every `message/send` call returned a `Task` that was already `completed` (or `failed`). The agent did its work synchronously inside the request handler and the client got the answer in the same HTTP round-trip.

That's fine for fast lookups. Real agents do things that take seconds to minutes: hitting external APIs, running models, waiting on humans. A2A models that with a small but powerful state machine on the `Task` object itself, plus three methods that operate on long-running tasks:

- `tasks/get` — *"what's the status of task X right now?"*
- `tasks/cancel` — *"stop working on task X."*
- A second `message/send` with `taskId` set — *"here's the clarification you asked for on task X."*

This notebook builds a "slow researcher" that runs work in a background thread, exposes all four methods, and walks through every transition you can plausibly hit.

> *Targets A2A spec v0.3.0.*

## What you'll learn

- The A2A task state machine and which states are terminal vs. transient.
- How to implement a server that spawns background work and returns a `submitted` task immediately.
- How `tasks/get` lets a client poll for progress.
- How `tasks/cancel` stops in-flight work cleanly.
- How a server requests more information mid-task with the `input-required` state, and how the client replies with a follow-up `message/send` whose `Message.taskId` references the original task.
- Why polling is fundamentally wasteful — setting up notebook 05's streaming.